# 🧪 Phase 1 — Layer-Level GPU Tests
## Phoenix & Routed DeltaNet

**Goal:** Verify that both custom attention layers:
- ✅ Run on GPU (real Triton kernels compile)
- ✅ Produce correct output shapes
- ✅ Have no NaN/Inf values
- ✅ Compute valid gradients (backward pass works)
- ✅ `chunk` and `fused_recurrent` modes produce consistent results

> **Before running:** Make sure you selected a GPU runtime.
> `Runtime → Change runtime type → T4 GPU`

---
## Step 1 — Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU found!\n"
        "Go to: Runtime → Change runtime type → T4 GPU, then reconnect."
    )

print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   PyTorch version: {torch.__version__}")

---
## Step 2 — Install the `flash-linear-attention` package

This installs directly from your GitHub repo so Colab gets your latest code.

In [ ]:
GITHUB_USER = "henrijn"
GITHUB_REPO = "independent-study-fla"
GITHUB_BRANCH = "main"  # change if your working branch differs

!pip install -q einops transformers
!pip install -q git+https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git@{GITHUB_BRANCH}

print("\n✅ Installation complete.")

---
## Step 3 — Test config

In [ ]:
import torch

# ── Tiny dimensions so tests run fast on a free T4 ──────────
BATCH       = 2
SEQ_LEN     = 128    # > 64 → forces 'chunk' mode in forward()
HIDDEN_SIZE = 64
NUM_HEADS   = 4      # head_k_dim = 64/4 = 16, num_slots defaults to 16
TOPK        = 4      # must be ≤ num_slots (16)
DTYPE       = torch.bfloat16
DEVICE      = torch.device("cuda")

print("Test config:")
print(f"  batch={BATCH}, seq={SEQ_LEN}, hidden={HIDDEN_SIZE}")
print(f"  num_heads={NUM_HEADS}, topk={TOPK}, dtype={DTYPE}")

---
## Step 4 — Helper functions

In [ ]:
def section(title):
    print("\n" + "=" * 55)
    print(f"  {title}")
    print("=" * 55)

def check(name, condition, msg=""):
    status = "✅ PASS" if condition else "❌ FAIL"
    print(f"  {status}  {name}" + (f"  →  {msg}" if msg else ""))
    return condition

def assert_no_nan_inf(tensor, name):
    has_nan = torch.isnan(tensor).any().item()
    has_inf = torch.isinf(tensor).any().item()
    ok = not has_nan and not has_inf
    msg = ("contains NaN " if has_nan else "") + ("contains Inf" if has_inf else "")
    check(f"{name} — no NaN/Inf", ok, msg.strip())
    return ok

print("✅ Helper functions defined.")

---
## Test 1 — PhoenixAttention (chunk mode)

Tests: forward shape, NaN check, backward/gradients, eval determinism.

In [ ]:
from fla.layers.phoenix import PhoenixAttention

section("Test 1a — PhoenixAttention [chunk] forward")

layer_phoenix = PhoenixAttention(
    mode="chunk",
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    topk=TOPK,
    layer_idx=0,
    router_noise=False,   # deterministic
).to(device=DEVICE, dtype=DTYPE)
layer_phoenix.eval()

x = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)

with torch.no_grad():
    o, _, _ = layer_phoenix(x)

expected = (BATCH, SEQ_LEN, HIDDEN_SIZE)
check("Output shape", tuple(o.shape) == expected, f"got {tuple(o.shape)}, expected {expected}")
assert_no_nan_inf(o, "Output")

In [ ]:
section("Test 1b — PhoenixAttention [chunk] short-seq (forces fused_recurrent path)")

x_short = torch.randn(BATCH, 32, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o_short, _, _ = layer_phoenix(x_short)

check("Short-seq output shape", o_short.shape == (BATCH, 32, HIDDEN_SIZE), f"got {tuple(o_short.shape)}")
assert_no_nan_inf(o_short, "Short-seq output")

In [ ]:
section("Test 1c — PhoenixAttention [chunk] backward pass")

layer_phoenix.train()
x_grad = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE, requires_grad=True)
o_grad, _, _ = layer_phoenix(x_grad)
o_grad.sum().backward()

check("Input grad exists", x_grad.grad is not None)
if x_grad.grad is not None:
    assert_no_nan_inf(x_grad.grad, "Input grad")

all_ok = True
for name, p in layer_phoenix.named_parameters():
    if p.requires_grad:
        if p.grad is None:
            check(f"Param grad: {name}", False, "grad is None")
            all_ok = False
        elif not torch.isfinite(p.grad).all():
            check(f"Param grad finite: {name}", False, "non-finite")
            all_ok = False
if all_ok:
    check("All parameter grads finite", True)

In [ ]:
section("Test 1d — PhoenixAttention eval determinism")

layer_phoenix.eval()
x_test = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o1, _, _ = layer_phoenix(x_test)
    o2, _, _ = layer_phoenix(x_test)

check("Deterministic in eval", torch.allclose(o1, o2), "Two identical forward passes gave different output")

---
## Test 2 — PhoenixAttention (fused_recurrent mode)

Same checks but forces `fused_recurrent` mode explicitly.

In [ ]:
section("Test 2 — PhoenixAttention [fused_recurrent]")

layer_phoenix_fr = PhoenixAttention(
    mode="fused_recurrent",
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    topk=TOPK,
    layer_idx=0,
    router_noise=False,
).to(device=DEVICE, dtype=DTYPE)
layer_phoenix_fr.eval()

x = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o, _, _ = layer_phoenix_fr(x)

check("Output shape", tuple(o.shape) == (BATCH, SEQ_LEN, HIDDEN_SIZE), f"got {tuple(o.shape)}")
assert_no_nan_inf(o, "Output")

# backward
layer_phoenix_fr.train()
x_grad = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE, requires_grad=True)
o_grad, _, _ = layer_phoenix_fr(x_grad)
o_grad.sum().backward()
check("Input grad exists", x_grad.grad is not None)
if x_grad.grad is not None:
    assert_no_nan_inf(x_grad.grad, "Input grad")
check("All param grads finite",
      all(p.grad is not None and torch.isfinite(p.grad).all()
          for p in layer_phoenix_fr.parameters() if p.requires_grad))

---
## Test 3 — RoutedDeltaNetAttention (chunk mode)

In [ ]:
from fla.layers.routed_deltanet import RoutedDeltaNetAttention

section("Test 3a — RoutedDeltaNetAttention [chunk] forward")

layer_rdn = RoutedDeltaNetAttention(
    mode="chunk",
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    topk=TOPK,
    layer_idx=0,
    router_noise=False,
).to(device=DEVICE, dtype=DTYPE)
layer_rdn.eval()

x = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o, _, _ = layer_rdn(x)

check("Output shape", tuple(o.shape) == (BATCH, SEQ_LEN, HIDDEN_SIZE), f"got {tuple(o.shape)}")
assert_no_nan_inf(o, "Output")

In [ ]:
section("Test 3b — RoutedDeltaNetAttention [chunk] short-seq")

x_short = torch.randn(BATCH, 32, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o_short, _, _ = layer_rdn(x_short)

check("Short-seq output shape", o_short.shape == (BATCH, 32, HIDDEN_SIZE), f"got {tuple(o_short.shape)}")
assert_no_nan_inf(o_short, "Short-seq output")

In [ ]:
section("Test 3c — RoutedDeltaNetAttention [chunk] backward")

layer_rdn.train()
x_grad = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE, requires_grad=True)
o_grad, _, _ = layer_rdn(x_grad)
o_grad.sum().backward()

check("Input grad exists", x_grad.grad is not None)
if x_grad.grad is not None:
    assert_no_nan_inf(x_grad.grad, "Input grad")

all_ok = True
for name, p in layer_rdn.named_parameters():
    if p.requires_grad:
        if p.grad is None:
            check(f"Param grad: {name}", False, "grad is None")
            all_ok = False
        elif not torch.isfinite(p.grad).all():
            check(f"Param grad finite: {name}", False, "non-finite")
            all_ok = False
if all_ok:
    check("All parameter grads finite", True)

In [ ]:
section("Test 3d — RoutedDeltaNetAttention use_gate=True variant")

layer_rdn_gated = RoutedDeltaNetAttention(
    mode="chunk",
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    topk=TOPK,
    layer_idx=0,
    router_noise=False,
    use_gate=True,
).to(device=DEVICE, dtype=DTYPE)
layer_rdn_gated.eval()

x = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o_gated, _, _ = layer_rdn_gated(x)

check("Gated output shape", tuple(o_gated.shape) == (BATCH, SEQ_LEN, HIDDEN_SIZE), f"got {tuple(o_gated.shape)}")
assert_no_nan_inf(o_gated, "Gated output")

---
## Test 4 — RoutedDeltaNetAttention (fused_recurrent mode)

In [ ]:
section("Test 4 — RoutedDeltaNetAttention [fused_recurrent]")

layer_rdn_fr = RoutedDeltaNetAttention(
    mode="fused_recurrent",
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    topk=TOPK,
    layer_idx=0,
    router_noise=False,
).to(device=DEVICE, dtype=DTYPE)
layer_rdn_fr.eval()

x = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o, _, _ = layer_rdn_fr(x)

check("Output shape", tuple(o.shape) == (BATCH, SEQ_LEN, HIDDEN_SIZE), f"got {tuple(o.shape)}")
assert_no_nan_inf(o, "Output")

# backward
layer_rdn_fr.train()
x_grad = torch.randn(BATCH, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE, requires_grad=True)
o_grad, _, _ = layer_rdn_fr(x_grad)
o_grad.sum().backward()
check("Input grad exists", x_grad.grad is not None)
if x_grad.grad is not None:
    assert_no_nan_inf(x_grad.grad, "Input grad")
check("All param grads finite",
      all(p.grad is not None and torch.isfinite(p.grad).all()
          for p in layer_rdn_fr.parameters() if p.requires_grad))

---
## Test 5 — chunk vs fused_recurrent consistency

Both modes should produce numerically similar outputs given the same weights and input.  
Large divergence here means a bug in one of the Triton kernels.

In [ ]:
section("Test 5a — Phoenix: chunk ≈ fused_recurrent")

layer_c = PhoenixAttention(
    mode="chunk", hidden_size=HIDDEN_SIZE, num_heads=NUM_HEADS,
    topk=TOPK, layer_idx=0, router_noise=False,
).to(device=DEVICE, dtype=DTYPE)
layer_c.eval()

layer_r = PhoenixAttention(
    mode="fused_recurrent", hidden_size=HIDDEN_SIZE, num_heads=NUM_HEADS,
    topk=TOPK, layer_idx=0, router_noise=False,
).to(device=DEVICE, dtype=DTYPE)
layer_r.eval()

# Share exact same weights
layer_r.load_state_dict(layer_c.state_dict())

x = torch.randn(1, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o_chunk, _, _ = layer_c(x)
    o_recur, _, _ = layer_r(x)

atol = 5e-2  # bfloat16 has low precision, ~0.05 is reasonable
max_diff = (o_chunk - o_recur).abs().max().item()
close = torch.allclose(o_chunk, o_recur, atol=atol, rtol=1e-2)
check(f"chunk ≈ fused_recurrent (max_diff={max_diff:.4f}, atol={atol})", close,
      f"max absolute difference = {max_diff:.4f}")

In [ ]:
section("Test 5b — RoutedDeltaNet: chunk ≈ fused_recurrent")

layer_c = RoutedDeltaNetAttention(
    mode="chunk", hidden_size=HIDDEN_SIZE, num_heads=NUM_HEADS,
    topk=TOPK, layer_idx=0, router_noise=False,
).to(device=DEVICE, dtype=DTYPE)
layer_c.eval()

layer_r = RoutedDeltaNetAttention(
    mode="fused_recurrent", hidden_size=HIDDEN_SIZE, num_heads=NUM_HEADS,
    topk=TOPK, layer_idx=0, router_noise=False,
).to(device=DEVICE, dtype=DTYPE)
layer_r.eval()

layer_r.load_state_dict(layer_c.state_dict())

x = torch.randn(1, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    o_chunk, _, _ = layer_c(x)
    o_recur, _, _ = layer_r(x)

atol = 5e-2
max_diff = (o_chunk - o_recur).abs().max().item()
close = torch.allclose(o_chunk, o_recur, atol=atol, rtol=1e-2)
check(f"chunk ≈ fused_recurrent (max_diff={max_diff:.4f}, atol={atol})", close,
      f"max absolute difference = {max_diff:.4f}")

---
## 🏁 Summary

Run this cell last to see a clean pass/fail overview.

In [ ]:
print("\n" + "="*55)
print("  PHASE 1 COMPLETE")
print("="*55)
print("""
Scroll up and look for any  ❌ FAIL  lines.

If all ✅ PASS  → move on to Phase 2 (full model forward pass).
If any ❌ FAIL  → copy the error message and go fix it in Antigravity,
                  then push to GitHub and re-run this notebook.
""")